In [1]:
# 加载环境变量
from dotenv import load_dotenv
from langchain_core.messages import AIMessage

load_dotenv()

True

# 定义模型
我们定义两个模型，一个多模态模型，一个聊天模型


In [2]:
from langchain.chat_models import init_chat_model
import os

# 多模态模型
multimodal_model = init_chat_model(
    model="qwen3-omni-flash",  # 模型名称，这里选择qwen3.5-plus，这是一个多模态模型，支持图片、文本、音频、视频
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

# 定义工具


In [3]:
from langchain_tavily import TavilySearch

# web搜索工具，使用tavily作为web搜索工具
web_search = TavilySearch(
    max_results=5,
    topic="general"
)

# 添加记忆管理

这里我们采用sqllite

In [4]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# 初始化checkpointer
checkpointer = SqliteSaver(sqlite3.connect("resources/personal_chief.db", check_same_thread=False))
# 自动建表
checkpointer.setup()

# 定义智能体

In [5]:
from langchain.agents import create_agent

system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，再搜索不到的情况下才能自己发挥。
"""

agent = create_agent(
    model=multimodal_model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

# 测试


In [6]:
from langchain.messages import HumanMessage

multimodal_message = HumanMessage(
    content=[
        {"type": "image",
         "url": "https://tmp9527.oss-cn-beijing.aliyuncs.com/1772678117299.jpeg?x-oss-signature-version=OSS4-HMAC-SHA256&x-oss-date=20260305T082142Z&x-oss-expires=299&x-oss-credential=LTAI5tHzyC36KhUJfPqibaCo%2F20260305%2Fcn-beijing%2Foss%2Faliyun_v4_request&x-oss-signature=90083bdacfacc36386069d1e82c33b04ccd8eaa71c43fcf9248a36535d9cf46a"},
        {"type": "text", "text": "帮我看看这些食材能做些什么？"}
    ])

config = {"configurable": {"thread_id": "1"}}

In [ ]:
response = agent.invoke({"messages": [multimodal_message]}, config)

In [53]:
# 友好打印
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_dd62e914d5d543379ec6aa)
 Call ID: call_dd62e914d5d543379ec6aa
  Args:
    query: 用西兰花、三文鱼、鸡胸肉、蘑菇、小番茄、红椒、生菜、白萝卜做的菜谱
    search_depth: advanced
================================= Tool Message =================================
Name: tavily_search

{"query": "用西兰花、三文鱼、鸡胸肉、蘑菇、小番茄、红椒、生菜、白萝卜做的菜谱", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://cookpad.com/tw/%E9%A3%9F%E8%AD%9C/13394466", "title": "三文魚&雞胸肉食譜與作法by Suky🤭", "content": "Cookpad\nCookpad\n訪客\nCookpad\nCookpad\nCookpad\n三文魚&雞胸肉 的食譜成品照片\n\n# 三文魚&雞胸肉\n\nSuky🤭\n\n健康的飲食\n  \n（配菜方面可以選擇自己鍾意既蔬菜）\n\n健康的飲食\n  \n（配菜方面可以選擇自己鍾意既蔬菜）\n\n# 三文魚&雞胸

In [54]:
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢第3道菜，可以说的更详细点吗？")]},
    config
)

In [55]:
# 友好打印
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

当然可以！既然您对 **“鸡胸肉西兰花炒饭”** 感兴趣，我会为您详细展开这道菜的制作过程，包括食材准备、步骤分解、调味技巧、搭配建议，甚至如何让这道菜更美味、更营养！以下是超详细的版本：

---

## 🍽️ **第3道菜：鸡胸肉西兰花炒饭（详细版）**

### 📌 **核心亮点**
- **高蛋白低脂**：鸡胸肉+西兰花，营养密度高。
- **色彩丰富**：红椒、小番茄、白萝卜、蘑菇，视觉上非常吸引人。
- **快手易做**：15-20分钟搞定，适合忙碌的上班族或学生党。
- **可搭配主食**：可单独吃，也可搭配糙米饭、藜麦饭、荞麦面等，营养更全面。

---

## 🛒 **食材清单（按份量）**

| 食材          | 用量（约2人份） | 备注 |
|---------------|------------------|------|
| 鸡胸肉        | 200g             | 建议选新鲜鸡胸肉，口感更嫩 |
| 西兰花        | 150g             | 选花蕾紧实、颜色翠绿的 |
| 白萝卜        | 100g             | 去皮切丁，可选嫩白萝卜 |
| 蘑菇（口蘑/香菇） | 80g             | 切片，增加鲜味 |
| 小番茄        | 4-5个            | 对半切，增加酸甜口感 |
| 红椒          | 1/2个            | 切丁，增加色彩和微辣风味 |
| 生菜          | 1小把（约50g）   | 最后加入，保持脆嫩 |
| 食用油        | 1-2汤匙          | 推荐橄榄油或菜籽油 |
| 调味料        | 盐、黑胡椒、酱油、料酒、蒜末（可选） | 基础调味，可按口味调整 |

---

## 🍳 **详细制作步骤**

### ✅ **第一步：准备食材（约5分钟）**
1. **鸡胸肉处理**：
   - 鸡胸肉洗净，用厨房纸吸干水分。
   - 切成1.5cm左右的丁，放入碗中。
   - 加入

# 流式调用

有工具调用的agent消息类型比较多，要正确的流式输出比较麻烦，这里给出一个示例


In [7]:
from langchain.messages import AIMessageChunk

for chunk, metadata in agent.stream(
            {"messages": [multimodal_message]},
            config,
            stream_mode="messages"
        ):
            if isinstance(chunk, AIMessageChunk) and chunk.content:
                print(chunk.content, end="", flush=True)

根据您提供的食材（土豆、猪肉、胡萝卜、番茄、青椒、茄子、大蒜、胡椒粒），我为您整理了一份营养均衡、制作简单的菜谱推荐清单。这些菜谱结合了您现有的食材，兼顾美味与健康，适合日常家庭烹饪。

---

### 🍽️ **推荐菜谱排名（按营养与制作难度综合评分）**

#### **1. 地三鲜（土豆、茄子、青椒）——经典家常菜（推荐指数：★★★★★）**
- **推荐理由**：这道菜是经典的“地三鲜”，搭配土豆、茄子和青椒，口感丰富，营养均衡。加入猪肉和大蒜，味道更香浓。
- **制作难度**：中等（约30分钟）
- **适合人群**：家庭聚餐、下饭神器
- **做法简述**：
  1. 土豆、茄子、青椒切块，胡萝卜切丝，猪肉切丁。
  2. 热锅加油，先煎土豆至金黄，盛出备用。
  3. 同一锅中炒香蒜末、姜片，加入猪肉丁翻炒至变色。
  4. 加入茄子、青椒、胡萝卜丝，翻炒至软。
  5. 加入生抽、盐、糖调味，最后加入土豆块，翻炒均匀即可。
  6. 可撒上胡椒粒或葱花点缀。

#### **2. 肉末茄子煲（推荐指数：★★★★☆）**
- **推荐理由**：茄子吸汁入味，搭配肉末和番茄，汤汁浓郁，非常适合配米饭。
- **制作难度**：中等（约40分钟）
- **适合人群**：喜欢浓郁口味的人
- **做法简述**：
  1. 茄子切块，用盐腌制10分钟，挤干水分。
  2. 热锅加油，将茄子煎至表面微焦，盛出备用。
  3. 锅中留底油，炒香蒜末、姜片，加入肉末翻炒至变色。
  4. 加入番茄块，炒至出汁，再加入茄子块、胡萝卜丝。
  5. 加入生抽、盐、糖调味，加适量水，小火炖煮15分钟。
  6. 最后撒上胡椒粒，收汁即可。

#### **3. 番茄胡萝卜炒土豆丝（推荐指数：★★★★）**
- **推荐理由**：这道菜色彩鲜艳，口感清爽，适合喜欢清淡口味的人。
- **制作难度**：低（约20分钟）
- **适合人群**：素食者、减脂人群
- **做法简述**：
  1. 土豆切丝，胡萝卜切丝，番茄切块。
  2. 热锅加油，先炒香蒜末，加入土豆丝翻炒至断生。
  3. 加入胡萝卜丝、番茄块，翻炒至番茄出汁。
  4. 加入盐、糖调味，撒上胡椒粒，翻炒均匀即可。

#### **4. 青椒炒肉片配土豆（推荐指数：★★★☆）**
- **推荐理由**：这道

# 获取会话历史

既然会话记忆交给了checkpointer管理，我们自然可以从中获取会话历史


In [7]:
checkpointer.get(config)

{'v': 4,
 'ts': '2026-03-05T04:32:04.748794+00:00',
 'id': '1f1184c4-318f-6c21-8005-8152a8ca695c',
 'channel_values': {'messages': [HumanMessage(content=[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}], additional_kwargs={}, response_metadata={}, id='a29981d9-1396-4397-9c42-8a3d4322a691'),
   AIMessage(content='', additional_kwargs={}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'qwen3-omni-flash', 'model_provider': 'openai'}, id='lc_run--019cbc43-db26-71e0-8385-79361516f7a7', tool_calls=[{'name': 'tavily_search', 'args': {'query': '用西兰花、三文鱼、鸡胸肉、蘑菇、小番茄、红椒、生菜、白萝卜做的菜谱', 'search_depth': 'advanced'}, 'id': 'call_3971f410f521440fab0430', 'type': 'tool_call'}], invalid_tool_calls=[]),
   ToolMessage(content='{"query": "用西兰花、三文鱼、鸡胸肉、蘑菇、小番茄、红椒、生菜、白萝卜做的菜谱", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://cookpad.co

In [71]:
for m in checkpointer.get(config)['channel_values']['messages']:
    print(type(m))
    print(m)

<class 'langchain_core.messages.human.HumanMessage'>
content=[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}] additional_kwargs={} response_metadata={} id='8c0219c3-2c70-46bb-bfa2-400a516e0117'
<class 'langchain_core.messages.ai.AIMessage'>
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 2151, 'total_tokens': 2206, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': None, 'rejected_prediction_tokens': None, 'text_tokens': 55}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'image_tokens': 262, 'text_tokens': 1889}}, 'model_provider': 'openai', 'model_name': 'qwen3-omni-flash', 'system_fingerprint': None, 'id': 'chatcmpl-a5c8f901-eaa4-9903-96e4-334a9cb802da', 'finish_reason': 'tool_calls', 'logprobs': None} id='l

定义一个便捷的获取会话历史的函数：


In [78]:
def get_messages(thread_id: str) -> list[dict[str, str]]:
    """获取会话历史"""

    # 根据 thread_id 查询 checkpoint
    cp = checkpointer.get({"configurable": {"thread_id": thread_id}})

    # 如果不存在，返回空列表
    if not cp:
        return []

    # 安全获取 messages
    channel_values = cp.get("channel_values")
    if not channel_values:
        return []

    messages = channel_values.get("messages", [])
    if not messages:
        return []

    # 转换消息格式
    result = []
    for msg in messages:
        if not msg.content:
            continue

        if isinstance(msg, HumanMessage):
            result.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            result.append({"role": "assistant", "content": msg.content})

    return result

In [82]:
get_messages("1")

[]

# 清空会话历史


In [81]:
checkpointer.delete_thread("1")